# Farm Agent — 4. LLM-judge scores for saved answers (CPU, no GPU)

**Settings:** Accelerator = None · Internet = On. **Secret:** `GROQ_API_KEY` (Add-ons → Secrets).

Notebook 2 saves every generated answer to `results/finetune/generations_<variant>.jsonl`. This notebook
judges those answers without re-running any model, which is useful when Groq's daily token limit
(200K tokens/day for `gpt-oss-120b` on the free tier, ~550 tokens per judged answer) cut the GPU run short.
Rows that are already scored are skipped, so a run that stops on the rate limit can simply be re-run later.

In [ ]:
REPO = "https://github.com/shashank-sj/farm-agent.git"
BRANCH = "main"
LABELS = "lora full qlora+rag"   # variants whose judge scores are missing or incomplete
# Code source: a private Kaggle dataset with the repo (attached as input) if present, else GitHub.
import glob, os, shutil, tarfile, zipfile
DST = "/tmp/farm-agent"
shutil.rmtree(DST, ignore_errors=True)
apps = glob.glob("/kaggle/input/**/app.py", recursive=True)
archives = glob.glob("/kaggle/input/**/farm_agent_src.*", recursive=True)
if apps:
    shutil.copytree(os.path.dirname(apps[0]), DST)
elif archives:
    os.makedirs(DST)
    a = archives[0]
    (tarfile.open(a) if a.endswith((".tar.gz", ".tgz")) else zipfile.ZipFile(a)).extractall(DST)
else:
    !git clone --depth 1 -b $BRANCH $REPO $DST
%cd $DST
OUT = "/kaggle/working"
!pip install -q groq

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
print("GROQ_API_KEY: loaded")

In [ ]:
!python -m finetune.judge_generations --labels $LABELS
!python -m finetune.summarize

In [ ]:
!mkdir -p $OUT/finetune
!cp results/finetune/eval_*.json results/finetune/generations_*.jsonl results/finetune/SUMMARY.md $OUT/finetune/
!ls -la $OUT/finetune